In [ ]:
# =============================================================================
# CELL 1: IMPORTS & ENVIRONMENT SETUP
# =============================================================================

import os
import random
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter
import copy
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torchvision import transforms
from torchvision.transforms import functional as TF

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.preprocessing import label_binarize

import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

# =============================================================================
# CELL 2: SEED EVERYTHING & GPU CHECK
# =============================================================================

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {device}")

if torch.cuda.is_available():
    print(f"🔥 GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"✅ FP16 (Half Precision) Supported: True")
else:
    print("⚠️ No GPU detected. Training will be slow on CPU.")
# ══════════════════════════════════════════════════════════════
# Cell 1: Installs & Imports
# ══════════════════════════════════════════════════════════════


import os, glob, random, math, warnings, time, copy

seed_everything(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = './saved_models'
os.makedirs(SAVE_DIR, exist_ok=True)


# =============================================================================
# CELL 3: CREATE SORTED CSV FILES FOR EACH CLASS
# =============================================================================

DATA_DIR = '/kaggle/input/new-chest/An Image Dataset of Advanced Imaging Techniques for Lung Cancer Diagnosis/Lung Cancer Diagnosis Dataset/Dataset'
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 3
CLASS_NAMES = ['Bengin cases', 'Malignant cases', 'Normal cases']
CLASS_LABELS = ['Benign', 'Malignant', 'Normal']

TRAIN_RATIO = 0.70
VAL_RATIO = 0.10
TEST_RATIO = 0.20

def create_class_csv(data_dir, class_names, output_dir='.'):
    """Create sorted CSV files for each class with train/val/test split assignment."""
    os.makedirs(output_dir, exist_ok=True)
    
    all_data = []
    
    for cls_idx, cls_name in enumerate(class_names):
        cls_path = os.path.join(data_dir, cls_name)
        if not os.path.isdir(cls_path):
            print(f"⚠️ Directory not found: {cls_path}")
            continue
        
        # Get sorted file list
        files = sorted([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
        n_files = len(files)
        
        # Calculate split indices
        train_end = int(n_files * TRAIN_RATIO)
        val_end = int(n_files * (TRAIN_RATIO + VAL_RATIO))
        
        # Assign splits
        class_data = []
        for i, fname in enumerate(files):
            if i < train_end:
                split = 'train'
            elif i < val_end:
                split = 'val'
            else:
                split = 'test'
            
            class_data.append({
                'index': i + 1,
                'filename': fname,
                'filepath': os.path.join(cls_path, fname),
                'class_idx': cls_idx,
                'class_name': CLASS_LABELS[cls_idx],
                'split': split
            })
        
        # Save individual class CSV
        df_class = pd.DataFrame(class_data)
        csv_path = os.path.join(output_dir, f'{CLASS_LABELS[cls_idx].lower()}_dataset.csv')
        df_class.to_csv(csv_path, index=False)
        print(f"📁 {CLASS_LABELS[cls_idx]}: {n_files} images → {csv_path}")
        print(f"   Train: 1-{train_end} | Val: {train_end+1}-{val_end} | Test: {val_end+1}-{n_files}")
        
        all_data.extend(class_data)
    
    # Save combined CSV
    df_all = pd.DataFrame(all_data)
    combined_path = os.path.join(output_dir, 'full_dataset.csv')
    df_all.to_csv(combined_path, index=False)
    print(f"\n📊 Combined dataset saved: {combined_path}")
    
    return df_all

print("=" * 60)
print("📥 Creating Sorted CSV Files...")
print("=" * 60)

df_dataset = create_class_csv(DATA_DIR, CLASS_NAMES, output_dir='./dataset_csvs')

# Display split summary
print("\n" + "=" * 60)
print("📊 Split Summary from CSV:")
print("=" * 60)
for split in ['train', 'val', 'test']:
    split_df = df_dataset[df_dataset['split'] == split]
    print(f"\n{split.upper()}: {len(split_df)} images")
    print(split_df['class_name'].value_counts().to_string())

# =============================================================================
# CELL 4: LOAD DATA FROM CSV & APPLY CLAHE
# =============================================================================

def apply_clahe(image):
    """Apply CLAHE to enhance contrast."""
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    return enhanced

def load_images_from_csv(df, img_size=224):
    """Load and preprocess images based on CSV dataframe."""
    images = []
    labels = []
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Loading"):
        try:
            img = cv2.imread(row['filepath'])
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (img_size, img_size))
            img = apply_clahe(img)
            img = img.astype(np.float32) / 255.0
            images.append(img)
            labels.append(row['class_idx'])
        except Exception as e:
            print(f"Error loading {row['filepath']}: {e}")
            continue
    
    return np.array(images), np.array(labels)

print("\n" + "=" * 60)
print("📥 Loading Images from CSV...")
print("=" * 60)

# Load each split separately
df_train = df_dataset[df_dataset['split'] == 'train']
df_val = df_dataset[df_dataset['split'] == 'val']
df_test = df_dataset[df_dataset['split'] == 'test']

X_train, y_train = load_images_from_csv(df_train, IMG_SIZE)
X_val, y_val = load_images_from_csv(df_val, IMG_SIZE)
X_test, y_test = load_images_from_csv(df_test, IMG_SIZE)

print(f"\n✅ Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"📊 Train distribution: {dict(Counter(y_train))}")
print(f"📊 Val distribution:   {dict(Counter(y_val))}")
print(f"📊 Test distribution:  {dict(Counter(y_test))}")

# =============================================================================
# CELL 5: AUGMENT ONLY TRAINING X (BALANCED)
# =============================================================================

def augment_image(image_np):
    """Apply random augmentations."""
    img = Image.fromarray((image_np * 255).astype(np.uint8))
    
    # Random rotation
    angle = random.uniform(-20, 20)
    img = TF.rotate(img, angle, fill=0)
    
    # Random translation
    max_shift = int(IMG_SIZE * 0.1)
    dx = random.randint(-max_shift, max_shift)
    dy = random.randint(-max_shift, max_shift)
    img = TF.affine(img, angle=0, translate=(dx, dy), scale=1.0, shear=0)
    
    # Random shear
    shear_deg = random.uniform(-10, 10)
    img = TF.affine(img, angle=0, translate=(0, 0), scale=1.0, shear=shear_deg)
    
    # Random scale
    scale = random.uniform(0.9, 1.1)
    new_size = int(IMG_SIZE * scale)
    img = img.resize((new_size, new_size), Image.BILINEAR)
    img = TF.center_crop(img, [IMG_SIZE, IMG_SIZE])
    
    # Random horizontal flip
    if random.random() > 0.5:
        img = TF.hflip(img)
    
    return np.array(img).astype(np.float32) / 255.0

def augment_training_data(X_train, y_train, target_per_class=3333):
    """Augment training images to balance classes."""
    aug_images = []
    aug_labels = []
    
    for cls_idx in range(NUM_CLASSES):
        cls_mask = y_train == cls_idx
        cls_images = X_train[cls_mask]
        n_current = len(cls_images)
        n_needed = target_per_class - n_current
        
        # Add original images
        aug_images.extend(cls_images)
        aug_labels.extend([cls_idx] * n_current)
        
        # Generate augmented images
        print(f"  {CLASS_LABELS[cls_idx]}: {n_current} original + {max(0, n_needed)} augmented → {target_per_class}")
        
        for i in range(max(0, n_needed)):
            src_img = cls_images[i % n_current]
            aug_img = augment_image(src_img)
            aug_images.append(aug_img)
            aug_labels.append(cls_idx)
    
    return np.array(aug_images), np.array(aug_labels)

print("\n" + "=" * 60)
print("🔄 Augmenting Training Data Only...")
print("=" * 60)

TARGET_PER_CLASS = 3333
X_train_aug, y_train_aug = augment_training_data(X_train, y_train, TARGET_PER_CLASS)

print(f"\n✅ Augmented Training: {len(X_train_aug)} images")
print(f"📊 Balanced distribution: {dict(Counter(y_train_aug))}")
print(f"\n⚠️ Val/Test sets unchanged (no augmentation applied)")

# =============================================================================
# CELL 6: VISUALIZATION
# =============================================================================

fig, axes = plt.subplots(3, 6, figsize=(18, 10))
fig.suptitle('Preprocessed Training Samples (After CLAHE + Augmentation)', fontsize=16, fontweight='bold')

for row, cls_idx in enumerate(range(NUM_CLASSES)):
    cls_mask = y_train_aug == cls_idx
    cls_imgs = X_train_aug[cls_mask]
    sample_indices = np.random.choice(len(cls_imgs), 6, replace=False)
    
    for col, idx in enumerate(sample_indices):
        axes[row, col].imshow(cls_imgs[idx])
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(f'{CLASS_LABELS[cls_idx]}', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('preprocessed_samples.png', dpi=150, bbox_inches='tight')
plt.show()

# Class distribution comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original train
orig_dist = Counter(y_train)
axes[0].bar(CLASS_LABELS, [orig_dist[i] for i in range(3)], color=['#2ecc71', '#e74c3c', '#3498db'])
axes[0].set_title('Original Training Set', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate([orig_dist[i] for i in range(3)]):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

# Augmented train
aug_dist = Counter(y_train_aug)
axes[1].bar(CLASS_LABELS, [aug_dist[i] for i in range(3)], color=['#2ecc71', '#e74c3c', '#3498db'])
axes[1].set_title('Augmented Trainzing Set', fontweight='bold')
for i, v in enumerate([aug_dist[i] for i in range(3)]):
    axes[1].text(i, v + 50, str(v), ha='center', fontweight='bold')
    
# Test set
test_dist = Counter(y_test)
axes[2].bar(CLASS_LABELS, [test_dist[i] for i in range(3)], color=['#2ecc71', '#e74c3c', '#3498db'])
axes[2].set_title('Test Set (No Augmentation)', fontweight='bold')
for i, v in enumerate([test_dist[i] for i in range(3)]):
    axes[2].text(i, v + 2, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Ready for training!")
print(f"   X_train_aug: {X_train_aug.shape}")
print(f"   X_val: {X_val.shape}")
print(f"   X_test: {X_test.shape}")


# =============================================================================
# CELL 7: PYTORCH DATASET & DATALOADERS
# =============================================================================

class LungCancerDataset(Dataset):
    """Custom PyTorch Dataset for lung cancer CT images."""

    def __init__(self, images, labels, augment=False):
        self.images = images                                   # Store image array
        self.labels = labels                                   # Store label array
        self.augment = augment                                 # Flag for runtime augmentation

        self.transform = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.3),            # Occasional horizontal flip
            transforms.RandomRotation(10),                     # Small random rotation
        ]) if augment else None

    def __len__(self):
        return len(self.labels)                                # Total number of samples

    def __getitem__(self, idx):
        image = self.images[idx]                               # Get image at index [H,W,C]
        label = self.labels[idx]                               # Get label at index

        image = torch.tensor(image, dtype=torch.float32)       # Convert to float32 tensor
        image = image.permute(2, 0, 1)                         # [H,W,C] -> [C,H,W] for PyTorch

        if self.transform:                                     # Apply runtime augmentation if enabled
            image = self.transform(image)

        label = torch.tensor(label, dtype=torch.long)          # Convert label to long tensor
        return image, label

# --- Create Datasets (train uses augmented, val/test use original) ---
train_dataset = LungCancerDataset(X_train_aug, y_train_aug, augment=True)
val_dataset = LungCancerDataset(X_val, y_val, augment=False)      # Separate val dataset (no augmentation)
test_dataset = LungCancerDataset(X_test, y_test, augment=False)

# --- Create DataLoaders (NO random_split - sequential split already done) ---
NUM_WORKERS = 2

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

print(f"📦 Train loader: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"📦 Val loader:   {len(val_dataset)} samples, {len(val_loader)} batches")
print(f"📦 Test loader:  {len(test_dataset)} samples, {len(test_loader)} batches")

In [10]:
# ══════════════════════════════════════════════════════════════
# Cell 6: Architecture — DSConv, Stem, ResBlock, Downsample
# ══════════════════════════════════════════════════════════════

class DSConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, kernel_size, stride, padding, groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=bias)
    def forward(self, x):
        return self.pw(self.dw(x))

class DSConvStem(nn.Module):
    def __init__(self, in_ch=3, mid_ch=32, out_ch=64):
        super().__init__()
        self.stem = nn.Sequential(
            DSConv2d(in_ch, mid_ch, 3, stride=2, padding=1),
            nn.BatchNorm2d(mid_ch), nn.GELU(),
            DSConv2d(mid_ch, out_ch, 3, stride=2, padding=1),
            nn.BatchNorm2d(out_ch), nn.GELU(),
        )
    def forward(self, x):
        return self.stem(x)

class DSConvResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            DSConv2d(channels, channels, 3, 1, 1),
            nn.BatchNorm2d(channels), nn.GELU(),
            DSConv2d(channels, channels, 3, 1, 1),
            nn.BatchNorm2d(channels),
        )
    def forward(self, x):
        return self.block(x) + x

class DSConvDownsample(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.down = nn.Sequential(
            DSConv2d(in_ch, out_ch, 3, stride=2, padding=1),
            nn.BatchNorm2d(out_ch),
        )
    def forward(self, x):
        return self.down(x)

print("✅ Building blocks defined")
# ══════════════════════════════════════════════════════════════
# Cell 7: Mamba Mixer, Self-Attention, MLP Blocks
# ══════════════════════════════════════════════════════════════

class SelectiveScan(nn.Module):
    def __init__(self, d_inner, d_state=16):
        super().__init__()
        self.d_state = d_state
        self.dt_rank = max(1, d_inner // 16)
        self.x_proj = nn.Linear(d_inner, self.dt_rank + 2 * d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, d_inner, bias=True)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).expand(d_inner, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_inner))

    def forward(self, x):
        B_batch, D, L = x.shape
        x_seq = x.transpose(1, 2)
        x_dbl = self.x_proj(x_seq)
        dt_raw, B_param, C_param = x_dbl.split([self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt_raw))
        A = -torch.exp(self.A_log)
        dA = torch.exp(dt.unsqueeze(-1) * A)
        dBx = dt.unsqueeze(-1) * B_param.unsqueeze(2)
        h = torch.zeros(B_batch, D, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            h = dA[:, t] * h + dBx[:, t] * x_seq[:, t, :, None]
            yt = (h * C_param[:, t, None, :]).sum(-1)
            ys.append(yt)
        y = torch.stack(ys, dim=2)
        return y + self.D.unsqueeze(0).unsqueeze(-1) * x

class MambaVisionMixer(nn.Module):
    def __init__(self, dim, d_state=16, kernel_size=3):
        super().__init__()
        half = dim // 2
        self.proj_x = nn.Linear(dim, half, bias=False)
        self.conv1d_x = nn.Conv1d(half, half, kernel_size, padding='same', groups=half)
        self.ssm = SelectiveScan(half, d_state)
        self.proj_z = nn.Linear(dim, half, bias=False)
        self.conv1d_z = nn.Conv1d(half, half, kernel_size, padding='same', groups=half)
        self.out_proj = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        residual = x
        x = self.norm(x)
        x1 = self.proj_x(x).transpose(1, 2)
        x1 = F.silu(self.conv1d_x(x1))
        x1 = self.ssm(x1)
        x2 = self.proj_z(x).transpose(1, 2)
        x2 = F.silu(self.conv1d_z(x2))
        out = torch.cat([x1, x2], dim=1).transpose(1, 2)
        return self.out_proj(out) + residual

class SelfAttentionBlock(nn.Module):
    def __init__(self, dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
    def forward(self, x):
        normed = self.norm(x)
        out, _ = self.attn(normed, normed, normed)
        return out + x

class MLPBlock(nn.Module):
    def __init__(self, dim, expand=4, dropout=0.1):
        super().__init__()
        hidden = dim * expand
        self.net = nn.Sequential(
            nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden, dim), nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x) + x

class MambaVisionLayer(nn.Module):
    def __init__(self, dim, mixer_type='mamba', num_heads=4, d_state=16, mlp_expand=4, dropout=0.1):
        super().__init__()
        if mixer_type == 'mamba':
            self.mixer = MambaVisionMixer(dim, d_state=d_state)
        else:
            self.mixer = SelfAttentionBlock(dim, num_heads=num_heads, dropout=dropout)
        self.mlp = MLPBlock(dim, expand=mlp_expand, dropout=dropout)
    def forward(self, x):
        x = self.mixer(x)
        x = self.mlp(x)
        return x

print("✅ Mamba/Attention/MLP blocks defined")
# ══════════════════════════════════════════════════════════════
# Cell 8: Complete DSMaT Architecture
# ══════════════════════════════════════════════════════════════

class DSMaT(nn.Module):
    def __init__(self, stem_channels=(32, 64), stage_channels=(96, 192, 256, 384),
                 stage_depths=(2, 2, 4, 4), sa_heads=(4, 8), d_state=16,
                 mlp_expand=4, num_classes=3, dropout=0.2):
        super().__init__()
        stem_mid, stem_out = stem_channels

        self.stem = DSConvStem(3, stem_mid, stem_out)

        self.down1 = DSConvDownsample(stem_out, stage_channels[0])
        self.stage1 = nn.Sequential(*[DSConvResBlock(stage_channels[0]) for _ in range(stage_depths[0])])

        self.down2 = DSConvDownsample(stage_channels[0], stage_channels[1])
        self.stage2 = nn.Sequential(*[DSConvResBlock(stage_channels[1]) for _ in range(stage_depths[1])])

        self.down3 = DSConvDownsample(stage_channels[1], stage_channels[2])
        n3 = stage_depths[2]; n3m = n3 // 2; n3a = n3 - n3m
        s3 = []
        for _ in range(n3m):
            s3.append(MambaVisionLayer(stage_channels[2], 'mamba', d_state=d_state,
                                       mlp_expand=mlp_expand, dropout=dropout))
        for _ in range(n3a):
            s3.append(MambaVisionLayer(stage_channels[2], 'attention', num_heads=sa_heads[0],
                                       mlp_expand=mlp_expand, dropout=dropout))
        self.stage3 = nn.ModuleList(s3)

        self.down4 = DSConvDownsample(stage_channels[2], stage_channels[3])
        n4 = stage_depths[3]; n4m = n4 // 2; n4a = n4 - n4m
        s4 = []
        for _ in range(n4m):
            s4.append(MambaVisionLayer(stage_channels[3], 'mamba', d_state=d_state,
                                       mlp_expand=mlp_expand, dropout=dropout))
        for _ in range(n4a):
            s4.append(MambaVisionLayer(stage_channels[3], 'attention', num_heads=sa_heads[1],
                                       mlp_expand=mlp_expand, dropout=dropout))
        self.stage4 = nn.ModuleList(s4)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.LayerNorm(stage_channels[3]),
            nn.Dropout(dropout),
            nn.Linear(stage_channels[3], num_classes)
        )

    def forward(self, x):
        B = x.shape[0]
        x = self.stem(x)
        x = self.down1(x); x = self.stage1(x)
        x = self.down2(x); x = self.stage2(x)
        x = self.down3(x)
        B3, C3, H3, W3 = x.shape
        x = x.flatten(2).transpose(1, 2)
        for layer in self.stage3:
            x = layer(x)
        x = x.transpose(1, 2).reshape(B, C3, H3, W3)
        x = self.down4(x)
        B4, C4, H4, W4 = x.shape
        x = x.flatten(2).transpose(1, 2)
        for layer in self.stage4:
            x = layer(x)
        x = x.transpose(1, 2).reshape(B, C4, H4, W4)
        x = self.pool(x).flatten(1)
        return self.head(x)

print("✅ DSMaT model defined")


✅ Building blocks defined
✅ Mamba/Attention/MLP blocks defined
✅ DSMaT model defined


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 9: Model Configurations — Pico & Nano Only
# ══════════════════════════════════════════════════════════════
DEVICE = "cuda"
MODEL_CONFIGS = {
    'DepthWiseMamba-Pico_50_epoch_lr_0.0001_new_dataset': dict(
        stem_channels=(16, 32), stage_channels=(48, 96, 128, 192),
        stage_depths=(1, 1, 2, 2), sa_heads=(4, 4), mlp_expand=2, dropout=0.2,
    ),
    'DepthwiseMamba-Nano_50_epoch_lr_0.0001_new_dataset': dict(
        stem_channels=(24, 48), stage_channels=(64, 128, 192, 256),
        stage_depths=(1, 2, 2, 2), sa_heads=(4, 4), mlp_expand=3, dropout=0.2,
    ),
}

def build_model(name, num_classes=3):
    cfg = {k: v for k, v in MODEL_CONFIGS[name].items()}
    return DSMaT(num_classes=num_classes, **cfg).to(DEVICE)

for name in MODEL_CONFIGS:
    m = build_model(name)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"{name:20s} | Params: {n_params:>10,} ({n_params/1e6:.2f}M)")
    del m


# ══════════════════════════════════════════════════════════════
# Cell 10: Training & Evaluation Utilities — WITH AUTO-RESUME
# FIX: Skips training if a saved model is found.
# ══════════════════════════════════════════════════════════════

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        loss_sum += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return loss_sum / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss_sum += loss.item() * labels.size(0)
        probs = F.softmax(outputs, dim=1)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    return (loss_sum / total, correct / total,
            np.array(all_preds), np.array(all_labels), np.array(all_probs))

# ---------------------------------------------------------
# 2. New Plotting Function
# ---------------------------------------------------------

def plot_history(history, model_name="Model"):
    """
    Plots training and validation loss and accuracy from the history dictionary.
    """
    # Check if history is empty
    if not history or not history.get('train_loss'):
        print("⚠️ No history data found to plot.")
        return

    epochs_range = range(1, len(history['train_loss']) + 1)
    
    plt.figure(figsize=(12, 5))

    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, history['train_loss'], label='Train Loss')
    plt.plot(epochs_range, history['val_loss'], label='Val Loss')
    plt.title(f'{model_name} Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Plot Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, history['train_acc'], label='Train Acc')
    plt.plot(epochs_range, history['val_acc'], label='Val Acc')
    plt.title(f'{model_name} Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------
# 3. Main Training Loop (Updated)
# ---------------------------------------------------------

def train_model(model, model_name, train_loader, val_loader, epochs=50, lr=1e-3, force_retrain=False):
    """
    Train with validation-based model selection. 
    Loads existing model if found, otherwise trains and saves best version.
    Automatically plots history at the end.
    """
    save_path = os.path.join(SAVE_DIR, f'{model_name}_best.pth')
    
    # ── CHECK FOR EXISTING MODEL ──
    if os.path.exists(save_path) and not force_retrain:
        print(f"\n✅ Found existing model for {model_name} at: {save_path}")
        print("   Skipping training and loading weights...")
        
        checkpoint = torch.load(save_path, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        # Retrieve history and best stats
        history = checkpoint.get('history', {})
        best_val_acc = checkpoint.get('val_acc', 0.0)
        best_epoch = checkpoint.get('epoch', '?')
        
        print(f"   Loaded! Best Val Acc: {best_val_acc*100:.2f}% (Epoch {best_epoch})")
        
        # Plot the loaded history
        plot_history(history, model_name)
        
        return history, best_val_acc, save_path

    # ── START TRAINING (if no model found) ──
    print(f"\n🚀 Starting training for {model_name}...")
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0

    for epoch in range(epochs):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_loss, val_acc, _, _, _ = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # ── SAVE BEST MODEL ──
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_loss': val_loss,
                'model_name': model_name,
                'history': history  # Saving history is crucial for plotting later!
            }, save_path)
            marker = ' ★ saved'
        else:
            marker = ''

        # Print EVERY epoch (Removed the modulo check)
        print(f"  Ep {epoch+1:3d}/{epochs} | Tr {tr_loss:.4f}/{tr_acc:.4f} "
              f"| Val {val_loss:.4f}/{val_acc:.4f} | {time.time()-t0:.1f}s{marker}")

    # ── RELOAD BEST MODEL from disk ──
    checkpoint = torch.load(save_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"\n  💾 Best model loaded from: {save_path}")
    print(f"     Best epoch: {checkpoint['epoch']} | Val Acc: {checkpoint['val_acc']*100:.2f}%")

    # Plot the training history we just finished
    plot_history(history, model_name)

    return history, best_val_acc, save_path

print("✅ Training utilities updated with Plotting & Full Epoch Logs")



In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 11: Train DSMaT-Pico and DSMaT-Nano
# ══════════════════════════════════════════════════════════════

EPOCHS = 50
LR = 0.0001
results = {}

for name in MODEL_CONFIGS.keys():
    print("=" * 70)
    print(f"  TRAINING: {name}")
    print("=" * 70)

    model = build_model(name)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Params: {n_params:,} ({n_params/1e6:.2f}M)")

    history, best_val_acc, save_path = train_model(
        model, name, train_loader, val_loader, epochs=EPOCHS, lr=LR
    )

    # Final evaluation on TEST set (only after training is complete)
    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc, test_preds, test_labels, test_probs = evaluate(
        model, test_loader, criterion, DEVICE
    )

    results[name] = {
        'history': history,
        'best_val_acc': best_val_acc,
        'test_acc': test_acc,
        'test_loss': test_loss,
        'test_preds': test_preds,
        'test_labels': test_labels,
        'test_probs': test_probs,
        'params': n_params,
        'save_path': save_path,
    }
    print(f"\n  ✅ Best Val Acc: {best_val_acc*100:.2f}% | Test Acc: {test_acc*100:.2f}%")
    print(f"  💾 Saved to: {save_path}\n")

    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

# ── Identify overall best model ──
best_model_name = max(results, key=lambda n: results[n]['test_acc'])
print("=" * 70)
print(f"🏆 BEST MODEL: {best_model_name} | Test Acc: {results[best_model_name]['test_acc']*100:.2f}%")
print(f"   Checkpoint: {results[best_model_name]['save_path']}")
print("=" * 70)


In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 13: Test-Set Evaluation — Confusion Matrix, Report, ROC
# ══════════════════════════════════════════════════════════════

class_labels = ['Benign', 'Malignant', 'Normal']

for name in MODEL_CONFIGS.keys():
    r = results[name]
    preds, labels, probs = r['test_preds'], r['test_labels'], r['test_probs']

    print("=" * 70)
    print(f"  TEST EVALUATION: {name} | Test Acc: {r['test_acc']*100:.2f}%")
    print(f"  Loaded from: {r['save_path']}")
    print("=" * 70)
    print(classification_report(labels, preds, target_names=class_labels))

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Confusion matrix
    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_labels, yticklabels=class_labels, ax=axes[0])
    axes[0].set_title(f'{name} — Confusion Matrix')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

    # ROC curves
    y_bin = label_binarize(labels, classes=[0, 1, 2])
    for cls_i in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(y_bin[:, cls_i], probs[:, cls_i])
        roc_auc = auc(fpr, tpr)
        axes[1].plot(fpr, tpr, label=f'{class_labels[cls_i]} (AUC={roc_auc:.3f})')
    axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
    axes[1].set_title(f'{name} — ROC Curves')
    axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f'test_eval_{name}.png', dpi=150, bbox_inches='tight')
    plt.show()